In [1]:
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
from tqdm import tqdm
import re
import ftfy
from collections import defaultdict

In [ ]:
# finePdf = pd.read_parquet('/Users/bramvanes/DATA/NLP/parquet/finepdf.parquet')

# finePdf = finePdf.assign(source='finepdf')
# finePdf = finePdf.assign(approx_token_counts_original=-1)
# finePdf = finePdf.rename(columns={'text_len':'approx_token_counts_translated'})
# finePdf.drop('score', axis=1, inplace=True)
# finePdf[['id', 'text', 'source', 'approx_token_counts_original', 'approx_token_counts_translated']]\
#                 .to_parquet('/Users/bramvanes/DATA/NLP/parquet/finepdf_clean.parquet', index=False)

In [ ]:
# fineWeb = pd.read_parquet('/Users/bramvanes/DATA/NLP/parquet/fineweb2_step2.parquet')

# fineWeb = fineWeb.assign(source='fineweb')
# fineWeb = fineWeb.assign(approx_token_counts_original=-1)
# fineWeb = fineWeb.rename(columns={'text_len':'approx_token_counts_translated'})
# fineWeb = fineWeb.drop('score', axis=1, inplace=True)
# fineWeb[['id', 'text', 'source', 'approx_token_counts_original', 'approx_token_counts_translated']]\
#                 .to_parquet('/Users/bramvanes/DATA/NLP/parquet/fineweb_clean.parquet', index=False)

# fineWeb.approx_token_counts_translated.sum()/1e9


In [ ]:

# (
#     pl.scan_parquet('/Users/bramvanes/DATA/NLP/parquet/fineweb2_step2.parquet')
#     .with_columns(
#         pl.lit('fineweb').alias('source'),
#         pl.lit(-1).alias('approx_token_counts_original'),
#         pl.lit(-1).alias('approx_token_counts_translated'),
#     )
#     .select(['id', 'text', 'source', 'approx_token_counts_original', 'approx_token_counts_translated'])
#     .sink_parquet(
#         '/Users/bramvanes/DATA/NLP/parquet/fineweb_clean.parquet',
#         row_group_size=100_000,  # optional: tune for your data
#     )
# )


In [ ]:
# use polars to load'/Users/bramvanes/DATA/NLP/parquet/fineweb_clean.parquet' in a dataframe

# fineweb = pl.read_parquet('/Users/bramvanes/DATA/NLP/parquet/fineweb_clean.parquet')

# # get the number of words in the text , split by space
# # get the number of words in the text, split by space
# fineweb = fineweb.with_columns(
#     pl.col('text').str.split(' ').list.len().alias('word_count')
# )

# fineweb['word_count'].cast(pl.Int64).sum()

In [ ]:
# I want to load the meddata.parquet file in a polars dataframe, lazy
#meddata= pl.read_parquet(r'T:\lab_research\RES-Folder-UPOD\CarTeksten\G_Output\2_Data\normalised\train\meddata.parquet', use_pyarrow=True, memory_map=True, low_memory=True)

In [ ]:

# (
#     pl.scan_parquet(r'T:\lab_research\RES-Folder-UPOD\CarTeksten\G_Output\2_Data\normalised\train\meddata.parquet')
#     .with_columns(
#         pl.lit('fineweb').alias('source'),
#         pl.lit(-1).alias('approx_token_counts_original'),
#         pl.lit(-1).alias('approx_token_counts_translated'),
#     )
#     .select(['id', 'text', 'source'])
# )

In [2]:
# TODO: Add more/improve cleaning steps as needed
RE_SPURIOUS_CHARS = re.compile(r'([^\w])\1{3,}')
RE_SPURIOUS_WORDS = re.compile(r'(\b[\w\-\s\;\:\,\.]+\b)\1{4,}')
RE_MULTISPACE = re.compile(r'\s{2,}')

def apply_until_stable(pattern, repl, text, max_iter=20):
    for _ in range(max_iter):
        text, changed = pattern.subn(repl, text)
        if changed == 0:
            break
    return text

def clean_text(text, num_reps=20):
    text = apply_until_stable(RE_SPURIOUS_WORDS, r'\1', text, num_reps)
    text = RE_SPURIOUS_CHARS.sub(r'\1', text)
    text = apply_until_stable(RE_SPURIOUS_WORDS, r'\1', text, num_reps)
    text = RE_MULTISPACE.sub(' ', text)
    text = ftfy.fix_encoding(text)
    return text

In [3]:
test = pd.read_parquet(r"T:\lab_research\RES-Folder-UPOD\CarTeksten\G_Output\2_Data\normalised\train\partitions\partition_000000.parquet")

In [ ]:
test['word_count'] = test.text.str.split(' ').apply(lambda x: len(x))

In [6]:
# iteratively process a parquet file in chunks of 100_000 rows, extract the word count of 'text' with split(" ") and give back the list of # of words
# Use pyarrow to read the parquet file in chunks and process it with tqdm for progress tracking
parquet_file = pq.ParquetFile(
    r'T:\lab_research\RES-Folder-UPOD\CarTeksten\G_Output\2_Data\normalised\train\meddata.parquet'
)

chunk_size = 100_000
word_lengths = []
total_count = 0
for batch in tqdm(parquet_file.iter_batches(batch_size=chunk_size, columns=['source', 'text']), total=parquet_file.num_row_groups):
    batch_dict = batch.to_pydict()
    sources = batch_dict['source']
    texts = batch_dict['text']

    for source, text in zip(sources, texts):
        wc = len(text.split())
        #cleaned_wc = len(clean_text(text, num_reps=7).split())
        word_lengths.append((source, wc))
        total_count += wc
    tqdm.write(f"Processed {total_count} words so far")

 34%|███▎      | 494/1471 [48:16<1:46:22,  6.53s/it]

Processed 23733067530 words so far


 34%|███▎      | 495/1471 [48:20<1:33:05,  5.72s/it]

Processed 23763480896 words so far


 34%|███▎      | 495/1471 [48:23<1:35:24,  5.87s/it]


KeyboardInterrupt: 

In [ ]:
source_counts = defaultdict(int)
source_len = defaultdict(int)
for source, wc in word_lengths:
    source_counts[source] += wc
    source_len[source] += 1
#sort the source counts by value
source_counts = dict(sorted(source_counts.items(), key=lambda item: item[1], reverse=True))
source_len = dict(sorted(source_len.items(), key=lambda item: item[1], reverse=True))

In [ ]:
# sum and len where mimic in key
keyword = '_comm'
sum(count for source, count in source_counts.items() if keyword in source.lower())/1e6, len([source for source in source_counts if keyword in source.lower()]), sum(count for source, count in source_len.items() if keyword in source.lower())/1e6